# Modul 8: LSTM dan GRU

**Nama:** ISI NAMA  
**NIM:** ISI NIM  
**Kelas:** ISI KELAS  
**Tanggal:** YYYY-MM-DD  

Simpan berkas ini sebagai `M08_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`; jangan menghapus sel pemeriksaan.
2. Gunakan `MODE_TUGAS=False` saat sesi 120 menit. Ubah menjadi `True` untuk hasil pengumpulan.
3. Eksperimen utama membandingkan RNN $H=228$, GRU $H=116$, dan LSTM $H=96$.
4. Semua model wajib memakai split, vocabulary, packing, optimizer, clipping, dan anggaran update yang sama.
5. Test set tidak digunakan untuk memilih model.
6. Luaran: `M08_NIM.ipynb`, `M08_NIM.pdf`, dan `M08_NIM_metrics.csv`.

**Bobot penilaian (total 100).** Pre-lab dan eksperimen retensi 15; state dan model generik 15 (Bagian B dan E); hitungan serta penyetaraan parameter 20; sembilan run multi-seed 30; visualisasi dan keputusan model 15 (Bagian G dan H); reproduksibilitas dan kerapian 5.


In [ ]:
import copy
import platform
import random
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

NIM = 'TODO'
BASE_SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42
MODE_TUGAS = False  # WAJIB True pada hasil pengumpulan
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAIN_N, VAL_N, EPOCHS = ((6_000, 1_500, 4) if MODE_TUGAS
                          else (2_400, 600, 2))
SEEDS = ([BASE_SEED, BASE_SEED + 1, BASE_SEED + 2]
         if MODE_TUGAS else [BASE_SEED])

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync_device() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

seed_everything(BASE_SEED)
pd.set_option('display.precision', 4)
print({'python': platform.python_version(), 'torch': torch.__version__,
       'device': str(DEVICE), 'mode_tugas': MODE_TUGAS,
       'train': TRAIN_N, 'validation': VAL_N, 'epochs': EPOCHS,
       'seeds': SEEDS})

## A. Pre-lab dan eksperimen retensi - 15 poin

1. **Mengapa hidden size yang sama tidak adil untuk RNN, GRU, dan LSTM?** TODO
2. **Apa perbedaan `h_n` dan `c_n` pada LSTM?** TODO
3. **Jika forget gate selalu 0,9, berapa informasi awal tersisa setelah 20 langkah?** TODO
4. **Mengapa test set tidak boleh dipakai untuk memilih sel atau hidden size?** TODO

### Kurva retensi cell state

In [ ]:
# TODO 1: plot f**T untuk f = 0.5, 0.9, 0.99 dan T=1..60.
# Laporkan nilai pada T=20 dan T=60 dalam DataFrame `retensi`.
raise NotImplementedError

assert set(retensi.columns) == {'forget_gate', 'retensi_T20', 'retensi_T60'}
assert np.isclose(retensi.loc[retensi['forget_gate'].eq(0.9),
                              'retensi_T20'].iloc[0], 0.9 ** 20)

## B. Bentuk state - bagian dari 15 poin

Komponen ini dinilai bersama Bagian E (model generik).


In [ ]:
# TODO 2: buat tensor x (B=4, T=12, E=16), panjang [12, 9, 6, 3],
# lalu jalankan nn.RNN, nn.GRU, dan nn.LSTM dengan H=32 melalui
# pack_padded_sequence. Cetak shape packed.data, h_n, dan c_n.
# Simpan ringkasan pada list `shape_rows` (satu dict per model).
raise NotImplementedError

shape_table = pd.DataFrame(shape_rows)
print(shape_table.to_string(index=False))
assert len(shape_table) == 3
assert shape_table.loc[shape_table['model'].eq('lstm'), 'c_n'].iloc[0] != 'tidak ada'
assert (shape_table.loc[~shape_table['model'].eq('lstm'), 'c_n'] == 'tidak ada').all()

**Interpretasi retensi dan state.** Jelaskan mengapa gerbang tidak berarti LSTM kebal terhadap vanishing gradient, serta mengapa GRU tidak memiliki `c_n`: TODO

## C. Hitungan dan penyetaraan parameter - 20 poin

In [ ]:
EMB, KELAS = 100, 4
FAKTOR = {'rnn': 1, 'gru': 3, 'lstm': 4}

def parameter_teori(jenis: str, hidden: int, emb: int = EMB, kelas: int = KELAS):
    # TODO 3: kembalikan (parameter_encoder, parameter_head, jumlahnya).
    # Encoder = faktor * H * (E + H + 2); head = H*C + C.
    raise NotImplementedError

def parameter_pytorch(jenis: str, hidden: int):
    # TODO 4: bentuk layer PyTorch dan kembalikan parameter encoder serta head.
    raise NotImplementedError

# TODO 5: buat `tabel_hidden_sama` untuk ketiga model pada H=96.
# Kolom minimal: model, hidden, encoder, head, encoder_plus_head.
raise NotImplementedError
print(tabel_hidden_sama.to_string(index=False))

for row in tabel_hidden_sama.itertuples():
    assert (row.encoder, row.head) == parameter_pytorch(row.model, row.hidden)

In [ ]:
target = parameter_teori('lstm', 96)[2]

def hidden_terdekat(jenis: str, target_budget: int, batas: int = 512):
    # TODO 6: cari integer 1..batas yang meminimalkan selisih absolut budget.
    raise NotImplementedError

KONFIG = {'rnn': hidden_terdekat('rnn', target),
          'gru': hidden_terdekat('gru', target),
          'lstm': 96}

# TODO 7: buat `tabel_budget` dengan kolom model, hidden, encoder, head,
# encoder_plus_head, dan selisih_pct terhadap target LSTM.
raise NotImplementedError
print(tabel_budget.to_string(index=False))

assert KONFIG == {'rnn': 228, 'gru': 116, 'lstm': 96}
assert tabel_budget['selisih_pct'].abs().max() <= 1.0
assert tabel_budget['encoder_plus_head'].between(76_000, 77_000).all()

**Bukti hitungan manual.** Tunjukkan satu penurunan lengkap untuk setiap jenis sel dan jelaskan mengapa embedding tidak dipakai untuk mencari hidden size: TODO

**Checkpoint menit ke-65:** tunjukkan dua angka retensi, tabel shape, kecocokan hitungan parameter, dan selisih anggaran maksimum 1%.

## D. Pipeline AG News

Sel berikut menyediakan pipeline Modul 7 agar waktu praktikum difokuskan pada sel bergerbang. Jangan mengubah split, sumber vocabulary, atau indeks token khusus.

In [ ]:
ROOT = Path('../../data/raw/ag_news')
if not ROOT.exists():
    ROOT = Path('data/raw/ag_news')
train_file = ROOT / 'train.csv'
if not train_file.exists():
    raise FileNotFoundError(
        f'{train_file} tidak ditemukan. Ikuti petunjuk data/README.md')

kolom = ['label', 'judul', 'ringkasan']
data = pd.read_csv(train_file, names=kolom, header=None)
if not str(data.iloc[0]['label']).strip().isdigit():
    data = data.iloc[1:].reset_index(drop=True)
data['teks'] = data['judul'].astype(str) + ' ' + data['ringkasan'].astype(str)
data['y'] = data['label'].astype(int) - 1

idx_train, idx_val = train_test_split(
    np.arange(len(data)), train_size=TRAIN_N, test_size=VAL_N,
    stratify=data['y'].to_numpy(), random_state=BASE_SEED)
teks_train = data['teks'].to_numpy()[idx_train]
teks_val = data['teks'].to_numpy()[idx_val]
y_train = data['y'].to_numpy()[idx_train]
y_val = data['y'].to_numpy()[idx_val]

POLA = re.compile(r"[a-z0-9']+")
def tokenisasi(teks):
    return POLA.findall(str(teks).lower())

cacah = Counter(token for teks in teks_train for token in tokenisasi(teks))
kosakata = ['<pad>', '<unk>'] + [w for w, n in cacah.most_common() if n >= 2]
stoi = {w: i for i, w in enumerate(kosakata)}
V = len(kosakata)
unk = sum(token not in stoi for teks in teks_val for token in tokenisasi(teks))
n_token = sum(len(tokenisasi(teks)) for teks in teks_val)

MAKS = 60
def ke_indeks(daftar_teks):
    X = torch.zeros(len(daftar_teks), MAKS, dtype=torch.long)
    L = torch.zeros(len(daftar_teks), dtype=torch.long)
    for i, teks in enumerate(daftar_teks):
        token = [stoi.get(t, 1) for t in tokenisasi(teks)][:MAKS] or [1]
        X[i, :len(token)] = torch.tensor(token); L[i] = len(token)
    return X, L

X_train, L_train = ke_indeks(teks_train)
X_val, L_val = ke_indeks(teks_val)
ds_train = TensorDataset(X_train, L_train, torch.tensor(y_train))
ds_val = TensorDataset(X_val, L_val, torch.tensor(y_val))

print({'train': len(ds_train), 'validation': len(ds_val), 'vocabulary': V,
       'validation_unk_pct': 100 * unk / n_token})
assert kosakata[:2] == ['<pad>', '<unk>']
assert sorted(np.unique(y_train)) == [0, 1, 2, 3]
assert L_train.min() >= 1 and L_train.max() <= MAKS

## E. Model generik - bagian dari 15 poin

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(self, jenis: str, hidden: int):
        super().__init__()
        # TODO 8: simpan jenis/hidden; buat Embedding(V, EMB, padding_idx=0),
        # encoder sesuai jenis, dan Linear(hidden, KELAS).
        raise NotImplementedError

    def forward(self, X, panjang):
        # TODO 9: embedding -> packing -> encoder -> h_n terakhir -> head.
        # Untuk LSTM, state adalah tuple (h_n, c_n).
        raise NotImplementedError

for jenis, hidden in KONFIG.items():
    model = SequenceClassifier(jenis, hidden)
    enc_n = sum(p.numel() for p in model.encoder.parameters())
    head_n = sum(p.numel() for p in model.head.parameters())
    assert (enc_n, head_n) == parameter_pytorch(jenis, hidden)
    with torch.no_grad():
        logits = model(X_train[:4], L_train[:4])
    assert logits.shape == (4, KELAS)
    print(jenis, hidden, enc_n + head_n, tuple(logits.shape))

## F. Pelatihan terkendali: sembilan run - 30 poin

Gunakan satu fungsi yang sama untuk seluruh model. Catat norma gradien sebelum clipping dan sinkronkan CUDA saat mengukur waktu.

In [ ]:
BATCH = 64

def buat_loader(ds, shuffle: bool, seed: int, batch: int = BATCH):
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, generator=generator)

@torch.no_grad()
def evaluasi(model, ds):
    # TODO 10: kembalikan loss rata-rata per contoh dan akurasi.
    raise NotImplementedError

def jalankan(jenis: str, hidden: int, seed: int):
    """TODO 11: latih satu run dan kembalikan model, history, row.

    Wajib: seed sebelum model/loader; Adam lr=1e-3; CrossEntropyLoss;
    packing; norma gradien sebelum clip_grad_norm_(..., 1.0); checkpoint
    berdasarkan validation loss; sync_device sebelum/sesudah timer.

    `history` memuat train_loss, val_loss, val_acc, grad_norm.
    `row` memuat seluruh kolom yang disebut pada modul, termasuk
    encoder_parameters, encoder_head_parameters, total_parameters,
    model_mib_fp32, n_updates, dan seconds_per_epoch.
    """
    raise NotImplementedError

In [ ]:
hasil, riwayat = [], {}
for seed in SEEDS:
    for jenis, hidden in KONFIG.items():
        print(f'Melatih {jenis.upper()} hidden={hidden}, seed={seed} ...')
        _, hist, row = jalankan(jenis, hidden, seed)
        hasil.append(row); riwayat[(jenis, seed)] = hist

tabel = pd.DataFrame(hasil)
kolom_tampil = ['model', 'hidden_size', 'seed', 'encoder_head_parameters',
                 'total_parameters', 'val_loss', 'val_accuracy',
                 'grad_norm_mean', 'seconds_per_epoch']
print(tabel[kolom_tampil].to_string(index=False))
assert len(tabel) == 3 * len(SEEDS)
assert tabel.groupby('seed')['n_updates'].nunique().max() == 1
assert (tabel['packing'] == True).all()
assert tabel['encoder_head_parameters'].between(76_000, 77_000).all()

## G. Visualisasi dan ringkasan - bagian dari 15 poin

In [ ]:
# TODO 12: dua panel berisi validation accuracy dan validation loss
# untuk tiga model pada BASE_SEED. Beri label sumbu, legenda, dan grid.
raise NotImplementedError

In [ ]:
# TODO 13: bentuk `ringkasan` per model yang memuat acc_mean, acc_std
# (ddof=0), seconds_mean, encoder_head, total_parameters, model_mib.
# Buat bar chart acc_mean dengan error bar satu SD dan anotasi waktu.
raise NotImplementedError

print(ringkasan.to_string(index=False))
assert len(ringkasan) == 3
if MODE_TUGAS:
    assert len(tabel) == 9 and tabel['seed'].nunique() == 3
    assert TRAIN_N == 6_000 and VAL_N == 1_500 and EPOCHS == 4

In [ ]:
if MODE_TUGAS:
    tabel.insert(1, 'student_id', NIM)
    output = Path(f'M08_{NIM}_metrics.csv')
    tabel.to_csv(output, index=False)
    print(f'{len(tabel)} baris disimpan ke {output}')
else:
    print('Quick mode: metrics tugas belum diekspor. Ubah MODE_TUGAS=True.')

## H. Pertanyaan analisis dan keputusan model - bagian dari 15 poin

1. Berapa parameter masing-masing model jika hidden size disamakan pada 96? Mengapa ini tidak adil? **TODO**
2. Setelah anggaran disetarakan, model mana yang memiliki rerata akurasi tertinggi? Apakah selisihnya melebihi variasi antar-seed? **TODO**
3. Model mana yang tercepat? Apakah urutannya sesuai jumlah gerbang dan parameter? **TODO**
4. Apa hubungan kurva $f^T$ dengan forget gate, dan mengapa LSTM masih dapat mengalami vanishing gradient? **TODO**
5. Pilih model untuk perangkat terbatas menggunakan minimal tiga angka dari hasil sendiri. **TODO**

## Checklist sebelum mengumpulkan

- [ ] Identitas terisi dan `MODE_TUGAS=True`.
- [ ] Protokol memakai 6.000 latih, 1.500 validasi, 4 epoch, dan 3 seed.
- [ ] Test set tidak dipakai untuk tuning.
- [ ] Hitungan manual cocok tepat dengan PyTorch.
- [ ] Selisih anggaran encoder + head maksimum 1%.
- [ ] Sembilan run tercatat, termasuk run yang kurang baik.
- [ ] Dua grafik dan ringkasan multi-seed berlabel lengkap.
- [ ] Semua jawaban analisis memakai angka hasil sendiri.
- [ ] `M08_NIM_metrics.csv` tersimpan.
- [ ] Notebook lolos *Restart Kernel and Run All*.